In [ ]:
!pip install vaderSentiment
!pip install transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 9.1 MB/s eta 0:00:00


In [ ]:
# Importación de librerías base
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

In [ ]:
# ─────────────────────────────────────────────
# 1A · Noticias desde yfinance (sin API key)
# ─────────────────────────────────────────────
def get_yfinance_news(ticker: str, max_items: int = 20) -> pd.DataFrame:
 """
 Descarga titulares recientes de Yahoo Finance para una acción.
 Parámetros:
 -----------
 ticker : str — símbolo bursátil (ej: 'XOM', 'MU', 'COST', 'JPM',)
 max_items: int — número máximo de noticias a traer
 Retorna:
 --------
 DataFrame con columnas: ticker, title, publisher, link, timestamp
 """
 stock = yf.Ticker(ticker)
 news = stock.news # Extraemos la lista de diccionarios con las noticias
 records = []
 for item in news[:max_items]:
    content = item.get('content', item)
    title = content.get('title', '') if isinstance(content, dict) else item.get('title', '')
    pub = content.get('provider', {})
    publisher = pub.get('displayName', '') if isinstance(pub, dict) else ''
    pub_time = content.get('pubDate', '') if isinstance(pub, dict) else ''
    link = content.get('canonicalUrl', {})
    url = link.get('url', '') if isinstance(link, dict) else ''
    records.append({
        'ticker' : ticker,
        'title' : title,
        'publisher' : publisher,
        'link' : url,
        'timestamp' : pub_time
    })
 df = pd.DataFrame(records)
 df = df[df['title'] != ''].reset_index(drop=True)
 return df

In [ ]:
# ─────────────────────────────────────────────
# 1B · Prueba con múltiples acciones
# ─────────────────────────────────────────────
TICKERS = ['XOM', 'MU', 'COST', 'JPM']
all_news = []
for tk in TICKERS:
 df_news = get_yfinance_news(tk, max_items=15)
 all_news.append(df_news)
 print(f"[{tk}] {len(df_news)} noticias descargadas exitosamente.")
df_news_all = pd.concat(all_news, ignore_index=True)
print(f"\nTotal de titulares recolectados: {len(df_news_all)}")
print(df_news_all[['ticker', 'title']].head(10))

[XOM] 10 noticias descargadas exitosamente.
[MU] 10 noticias descargadas exitosamente.
[COST] 10 noticias descargadas exitosamente.
[JPM] 10 noticias descargadas exitosamente.

Total de titulares recolectados: 40
  ticker                                              title
0    XOM  Trump meets with Brazil's Lula to talk trade, ...
1    XOM  Trump Admits “I Expected Oil to Hit $200” Over...
2    XOM  Oil Supply Shock Worsens amid Plunging Petrole...
3    XOM  Argus Hikes Exxon Mobil Price Target to $169 a...
4    XOM  Shell Earnings Surge on Iran War Oil Boom. Why...
5    XOM  Texas Pacific Land Corporation Q1 2026 Earning...
6    XOM  Drill, Baby, Drill! These 2 Oil Stocks Are Ram...
7    XOM  Shell Profits Climb as Iran War Boosts Oil Tra...
8    XOM  Exxon Mobil Uses AI Seismic Tools To Reshape G...
9    XOM                  3 Oil Stocks To Watch In May 2026


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import numpy as np

# ─────────────────────────────────────────────
# 2A · Inicializar VADER
# ─────────────────────────────────────────────
sia = SentimentIntensityAnalyzer()

# Demostración en vivo con frases financieras
demo_phrases = [
    "Apple beats earnings estimates, revenue surges to record high",
    "Fed signals aggressive rate hikes amid persistent inflation fears",
    "Stock market closed mixed, investors cautious ahead of jobs report",
    "Tesla faces regulatory probe over autopilot safety concerns",
    "Microsoft Cloud revenue grows 25% year over year, strong guidance"
]

print("═" * 65)
print(f"{'Frase':<52} | {'compound':>8}")
print("═" * 65)
for phrase in demo_phrases:
    scores = sia.polarity_scores(phrase)
    emoji  = "🟢" if scores['compound'] > 0.05 else ("🔴" if scores['compound'] < -0.05 else "⚪")
    print(f"{phrase[:50]:<52} | {scores['compound']:>7.3f} {emoji}")

═════════════════════════════════════════════════════════════════
Frase                                                | compound
═════════════════════════════════════════════════════════════════
Apple beats earnings estimates, revenue surges to    |   0.000 ⚪
Fed signals aggressive rate hikes amid persistent    |  -0.527 🔴
Stock market closed mixed, investors cautious ahea   |  -0.103 🔴
Tesla faces regulatory probe over autopilot safety   |   0.421 🟢
Microsoft Cloud revenue grows 25% year over year,    |   0.511 🟢


In [ ]:
# ─────────────────────────────────────────────
# 2B · Aplicar VADER a todo el dataset de noticias
# ─────────────────────────────────────────────
def apply_vader(df: pd.DataFrame, text_col: str = 'title') -> pd.DataFrame:
 """
 Aplica VADER sentiment a una columna de texto.
 Agrega columnas: neg, neu, pos, compound, sentiment_label
 """
 df = df.copy()
 scores = df[text_col].fillna('').apply(lambda t: sia.polarity_scores(t))
 df['vader_neg'] = scores.apply(lambda s: s['neg'])
 df['vader_neu'] = scores.apply(lambda s: s['neu'])
 df['vader_pos'] = scores.apply(lambda s: s['pos'])
 df['vader_compound'] = scores.apply(lambda s: s['compound'])
 df['sentiment_label'] = df['vader_compound'].apply(
 lambda c: 'positive' if c > 0.05 else ('negative' if c < -0.05 else
'neutral')
 )
 return df
df_vader = apply_vader(df_news_all, text_col='title')
print("\nDistribución de sentimiento VADER:")
print(df_vader.groupby(['ticker', 'sentiment_label']).size().unstack(fill_value=0))


Distribución de sentimiento VADER:
sentiment_label  negative  neutral  positive
ticker                                      
COST                    2        5         3
JPM                     3        3         4
MU                      2        5         3
XOM                     4        4         2


In [ ]:
# ─────────────────────────────────────────────
# 2C · Score diario agregado por acción
# ─────────────────────────────────────────────
def aggregate_sentiment_vader(df: pd.DataFrame) -> pd.DataFrame:
 """
 Calcula el score de sentimiento promedio por ticker.
 Score en [-1, +1]: +1 muy positivo, -1 muy negativo
 """
 agg = df.groupby('ticker').agg(
 n_noticias = ('vader_compound', 'count'),
 score_mean = ('vader_compound', 'mean'),
 score_std = ('vader_compound', 'std'),
 score_median = ('vader_compound', 'median'),
 pct_positive = ('sentiment_label', lambda x: (x == 'positive').mean() *
100),
 pct_negative = ('sentiment_label', lambda x: (x == 'negative').mean() *
100),
 pct_neutral = ('sentiment_label', lambda x: (x == 'neutral').mean() *
100)
 ).reset_index()
 agg['signal'] = agg['score_mean'].apply(
 lambda s: 'BULLISH' if s > 0.15 else ('BEARISH' if s < -0.15 else
'NEUTRAL')
 )
 return agg.sort_values('score_mean', ascending=False)
df_sentiment_vader = aggregate_sentiment_vader(df_vader)
print("\nScores VADER por acción:")
print(df_sentiment_vader[['ticker', 'score_mean', 'pct_positive', 'pct_negative', 'signal']].to_string(index=False))


Scores VADER por acción:
ticker  score_mean  pct_positive  pct_negative  signal
  COST     0.05327          30.0          20.0 NEUTRAL
   JPM     0.03345          40.0          30.0 NEUTRAL
    MU    -0.00030          30.0          20.0 NEUTRAL
   XOM    -0.20891          20.0          40.0 BEARISH


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

In [ ]:
# ─────────────────────────────────────────────
# 3A · Cargar modelo FinBERT
# NOTA: ~500 MB — pre-cargar antes de comenzar la clase
# ─────────────────────────────────────────────
MODEL_NAME = "ProsusAI/finbert"
print("Cargando FinBERT... (puede tardar en la primera ejecución)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval() # modo inferencia: sin gradientes
LABELS = ['positive', 'negative', 'neutral']
print("FinBERT cargado correctamente")

Cargando FinBERT... (puede tardar en la primera ejecución)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT cargado correctamente


In [ ]:
# ─────────────────────────────────────────────
# 3B · Función de inferencia FinBERT
# ─────────────────────────────────────────────
def finbert_sentiment(texts: list[str], batch_size: int = 8) -> list[dict]:
 """
 Aplica FinBERT a una lista de textos en batches.
 Retorna: [{'positive': p, 'negative': p, 'neutral': p,
 'compound': float, 'label': str}]
 """
 results = []
 for i in range(0, len(texts), batch_size):
  batch = texts[i : i + batch_size]
  inputs = tokenizer(
  batch, padding=True, truncation=True,
  max_length=512, return_tensors='pt'
  )
  with torch.no_grad():
   outputs = model(**inputs)
   probs = F.softmax(outputs.logits, dim=-1).numpy()
   for prob_row in probs:
    pos, neg, neu = prob_row[0], prob_row[1], prob_row[2]
    results.append({
    'positive' : round(float(pos), 4),
    'negative' : round(float(neg), 4),
    'neutral' : round(float(neu), 4),
    'compound' : round(float(pos - neg), 4),
    'label' : LABELS[int(prob_row.argmax())]})
 return results

In [ ]:
# ─────────────────────────────────────────────
# 3C · Aplicar FinBERT (máx. 50 noticias para clase)
# ─────────────────────────────────────────────
sample_df = df_news_all.copy().head(50)
texts_list = sample_df['title'].fillna('').tolist()
print(f"Analizando {len(texts_list)} titulares con FinBERT...")
finbert_results = finbert_sentiment(texts_list, batch_size=8)
sample_df['fb_positive'] = [r['positive'] for r in finbert_results]
sample_df['fb_negative'] = [r['negative'] for r in finbert_results]
sample_df['fb_neutral'] = [r['neutral'] for r in finbert_results]
sample_df['fb_compound'] = [r['compound'] for r in finbert_results]
sample_df['fb_label'] = [r['label'] for r in finbert_results]
print("FinBERT completado")
sample_df_vader = apply_vader(sample_df, 'title')
comparison = sample_df_vader[['ticker', 'title', 'vader_compound',
'fb_compound']].copy()
comparison['diferencia'] = (comparison['fb_compound'] -
comparison['vader_compound']).round(3)
comparison['acuerdo'] = (
 (comparison['vader_compound'] > 0.05) == (comparison['fb_compound'] > 0.05)
)
print("VADER vs FinBERT — muestra:")
print(comparison[['ticker', 'vader_compound', 'fb_compound',
'acuerdo']].head(15).to_string(index=False))
corr = comparison['vader_compound'].corr(comparison['fb_compound'])
acuerdo_pct = comparison['acuerdo'].mean() * 100
print(f"Correlación VADER-FinBERT: {corr:.3f}")
print(f"% titulares con mismo signo: {acuerdo_pct:.1f}%")

Analizando 40 titulares con FinBERT...
FinBERT completado
VADER vs FinBERT — muestra:
ticker  vader_compound  fb_compound  acuerdo
   XOM          0.1280       0.6121     True
   XOM         -0.4404       0.1120    False
   XOM         -0.6908      -0.9559     True
   XOM          0.0000       0.0417     True
   XOM         -0.6705      -0.8695     True
   XOM          0.0000      -0.0899     True
   XOM         -0.4926       0.1515    False
   XOM          0.0772       0.6591     True
   XOM          0.0000       0.2968    False
   XOM          0.0000       0.0166     True
    MU         -0.0772       0.5772    False
    MU          0.0000       0.0280     True
    MU         -0.5994      -0.0294     True
    MU          0.2023       0.6929     True
    MU          0.0000      -0.2010     True
Correlación VADER-FinBERT: 0.457
% titulares con mismo signo: 62.5%
